In [2]:

# Bicanski 2026 - UCM: universal cognitive maps
# https://doi.org/10.1016/j.cub.2026.08.064
# correspondence: bicanski@cbs.mpg.de

######

import numpy as np
import torch
from PIL import Image, ImageDraw
import torchvision.transforms as transforms
from pathlib import Path

def create_bird_image(leg_length, neck_length, image_size=64):   # for a single image
    
    img = Image.new('RGB', (image_size, image_size), 'white')
    draw = ImageDraw.Draw(img)
    
    # parameters
    center_x    = image_size // 2 - 5
    center_y    = image_size // 2
    body_x      = center_x
    body_y      = center_y + 2
    body_width  = 16
    body_height = 12
    
    # Draw body
    draw.ellipse([body_x - body_width//2, body_y - body_height//2, body_x + body_width//2, body_y + body_height//2], fill=(80, 80, 80))
    
    # Draw neck
    neck_start_x = body_x + body_width//2 - 1
    neck_start_y = body_y - 3
    neck_end_x   = neck_start_x
    neck_end_y   = neck_start_y - neck_length
    draw.line([neck_start_x, neck_start_y, neck_end_x, neck_end_y], fill=(100, 100, 100), width=3)
    
    # Draw head
    head_radius = 6
    head_x      = neck_end_x
    head_y      = neck_end_y
    draw.ellipse([head_x - head_radius, head_y - head_radius, head_x + head_radius, head_y + head_radius], fill=(100, 100, 100))
    
    # Draw eye
    eye_x = head_x + head_radius//2
    eye_y = head_y - head_radius//3
    draw.ellipse([eye_x - 1.5, eye_y - 1.5, eye_x + 1.5, eye_y + 1.5], fill=(235, 235, 235))
    
    # Draw beak
    beak_start_x = head_x + head_radius
    draw.polygon([(beak_start_x, head_y - 1),(beak_start_x + 4, head_y),(beak_start_x, head_y + 1)], fill=(255, 180, 50))
    
    # Draw legs
    front_leg_x = body_x + 2
    draw.line([front_leg_x, body_y + body_height//2, front_leg_x, body_y + body_height//2 + leg_length], fill=(70, 70, 70), width=2)
    
    back_leg_x = body_x - 2
    draw.line([back_leg_x, body_y + body_height//2, back_leg_x, body_y + body_height//2 + leg_length * 0.9], fill=(70, 70, 70), width=2)
    
    # Draw feet
    draw.line([front_leg_x, body_y + body_height//2 + leg_length,front_leg_x + 3, body_y + body_height//2 + leg_length], fill=(70, 70, 70), width=2)
    draw.line([back_leg_x, body_y + body_height//2 + leg_length * 0.9, back_leg_x - 3, body_y + body_height//2 + leg_length * 0.9], fill=(70, 70, 70), width=2)
    
    # Draw wing
    draw.ellipse([body_x - 4, body_y - 4, body_x + 4, body_y + 2], fill=(0, 0, 0))
    
    return img

In [3]:
repetitions = 50 #5 or 50

output_dir  = Path('datasets/birds')
# noise can be added after classification

# Define leg and neck length ranges (10 steps each)
leg_lengths  = np.linspace(6, 20, 16)
neck_lengths = np.linspace(5, 18, 16)

# Lists to store all images and labels
all_images      = []
all_leg_labels  = []
all_neck_labels = []

# Generate all combinations, repeated as specified
for rep in range(repetitions):
    for leg_idx, leg_length in enumerate(leg_lengths):
        for neck_idx, neck_length in enumerate(neck_lengths):

            img = create_bird_image(leg_length, neck_length)

            transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            ])
            img_tensor = transform(img)
            
            all_images.append(img_tensor)
            all_leg_labels.append(leg_idx)
            all_neck_labels.append(neck_idx)

# Convert to arrays
images_array = torch.stack(all_images).numpy()
leg_labels_array = np.array(all_leg_labels)
neck_labels_array = np.array(all_neck_labels)

# Save
if repetitions == 5:
    np.save(output_dir / 'bird_images.npy', images_array)
    np.save(output_dir / 'bird_leg_labels.npy', leg_labels_array)
    np.save(output_dir / 'bird_neck_labels.npy', neck_labels_array)
    np.save(output_dir / 'leg_lengths.npy', leg_lengths)
    np.save(output_dir / 'neck_lengths.npy', neck_lengths)

if repetitions == 50:
    np.save(output_dir / 'bird_images_Nlarge.npy', images_array)
    np.save(output_dir / 'bird_leg_labels_Nlarge.npy', leg_labels_array)
    np.save(output_dir / 'bird_neck_labels_Nlarge.npy', neck_labels_array)
    np.save(output_dir / 'leg_lengths_Nlarge.npy', leg_lengths)
    np.save(output_dir / 'neck_lengths_Nlarge.npy', neck_lengths)

print(f"Generated {len(all_images)} bird stimuli")

Generated 12800 bird stimuli
